# Exercice 2 — Segmentation intelligente des clients

**Objectif** : identifier automatiquement différents profils clients pour optimiser les campagnes marketing.

**Plan :**
1. Analyse exploratoire (EDA)
2. Nettoyage & feature engineering
3. Prétraitement (encodage, normalisation, PCA optionnelle)
4. Clustering (K-Means, DBSCAN, Agglomerative, GMM)
5. Évaluation des clusters (Silhouette, Elbow, Davies-Bouldin)
6. Interprétation métier des profils
7. Recommandations business

In [ ]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage

from src.preprocessing import (
    load_cluster_data, clean_customer_data,
    engineer_customer_features, encode_categorical, scale_features
)
from src.utils import plot_silhouette

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

DATA_PATH = '../data/raw/data_cluster.csv'
RANDOM_STATE = 42

## 1. Analyse exploratoire (EDA)

In [ ]:
df = load_cluster_data(DATA_PATH)
print(f"Dimensions : {df.shape}")
print(f"Colonnes : {df.columns.tolist()}")
df.head()

In [ ]:
print("--- Informations ---")
df.info()
print("\n--- Valeurs manquantes ---")
print(df.isnull().sum()[df.isnull().sum() > 0])
print("\n--- Statistiques ---")
df.describe()

In [ ]:
# Distribution des variables clés
key_vars = ['Income', 'Age' if 'Age' in df.columns else 'Year_Birth',
            'MntWines', 'MntMeatProducts', 'NumWebPurchases', 'NumStorePurchases']
key_vars = [v for v in key_vars if v in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, var in enumerate(key_vars[:6]):
    df[var].hist(bins=40, ax=axes[i], color='steelblue', edgecolor='white')
    axes[i].set_title(var)
    axes[i].set_xlabel('')

plt.suptitle('Distribution des variables principales', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig('../reports/figures/ex2_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Variables catégorielles
cat_vars = ['Education', 'Marital_Status']
cat_vars = [v for v in cat_vars if v in df.columns]

fig, axes = plt.subplots(1, len(cat_vars), figsize=(14, 5))
if len(cat_vars) == 1:
    axes = [axes]

for ax, var in zip(axes, cat_vars):
    df[var].value_counts().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title(var)
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../reports/figures/ex2_categorical.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Dépenses par canal d'achat
spend_cols = [c for c in df.columns if c.startswith('Mnt')]
channel_cols = [c for c in df.columns if 'Purchases' in c]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df[spend_cols].mean().sort_values(ascending=True).plot(
    kind='barh', ax=axes[0], color='steelblue'
)
axes[0].set_title('Dépenses moyennes par catégorie')

df[channel_cols].mean().sort_values(ascending=True).plot(
    kind='barh', ax=axes[1], color='coral'
)
axes[1].set_title('Achats moyens par canal')

plt.tight_layout()
plt.savefig('../reports/figures/ex2_spending_channels.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Nettoyage & Feature Engineering

In [ ]:
df_clean = clean_customer_data(df)
df_clean = engineer_customer_features(df_clean)

print(f"Lignes avant nettoyage : {len(df)}")
print(f"Lignes après nettoyage : {len(df_clean)}")
print(f"\nNouvelles features créées : Age, Children, TotalSpend, TotalPurchases")

In [ ]:
# Matrice de corrélation
num_cols = df_clean.select_dtypes(include=np.number).columns
corr_matrix = df_clean[num_cols].corr()

plt.figure(figsize=(16, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Matrice de corrélation', fontsize=14)
plt.tight_layout()
plt.savefig('../reports/figures/ex2_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Prétraitement pour clustering

In [ ]:
# Sélection des features pour le clustering
cat_cols = ['Education', 'Marital_Status']
drop_cols = ['ID', 'Dt_Customer', 'Year_Birth'] + cat_cols

df_encoded = encode_categorical(df_clean, cat_cols)
feature_cols = [c for c in df_encoded.select_dtypes(include=np.number).columns
                if c not in drop_cols]

X = df_encoded[feature_cols].copy()
print(f"Features pour clustering ({len(feature_cols)}) :\n{feature_cols}")

In [ ]:
X_scaled, scaler = scale_features(X)

# PCA pour visualisation 2D
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
print(f"Variance expliquée par les 2 premières composantes : {pca.explained_variance_ratio_.sum():.1%}")

# PCA pour réduction dimensionnelle (95% de variance)
pca_full = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_pca_full = pca_full.fit_transform(X_scaled)
print(f"Composantes pour 95% variance : {X_pca_full.shape[1]}")

In [ ]:
# Scree plot
pca_all = PCA(random_state=RANDOM_STATE)
pca_all.fit(X_scaled)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, len(pca_all.explained_variance_ratio_) + 1),
             pca_all.explained_variance_ratio_, 'o-', color='steelblue')
axes[0].set_title('Scree Plot')
axes[0].set_xlabel('Composante')
axes[0].set_ylabel('Variance expliquée')

axes[1].plot(range(1, len(pca_all.explained_variance_ratio_) + 1),
             np.cumsum(pca_all.explained_variance_ratio_), 'o-', color='coral')
axes[1].axhline(0.95, color='gray', linestyle='--', label='95%')
axes[1].set_title('Variance cumulative')
axes[1].set_xlabel('Composante')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/ex2_pca_scree.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Clustering

In [ ]:
# === K-Means : méthode Elbow + Silhouette ===
k_range = range(2, 11)
inertias = []
silhouette_scores = []
db_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_pca_full)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_pca_full, labels))
    db_scores.append(davies_bouldin_score(X_pca_full, labels))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(list(k_range), inertias, 'o-', color='steelblue')
axes[0].set_title('Elbow Method (Inertie)')
axes[0].set_xlabel('k')
axes[0].set_ylabel('Inertie')

plot_silhouette(silhouette_scores, list(k_range), ax=axes[1])

axes[2].plot(list(k_range), db_scores, 'o-', color='coral')
best_db_k = list(k_range)[np.argmin(db_scores)]
axes[2].axvline(best_db_k, color='gray', linestyle='--', label=f'Meilleur k={best_db_k}')
axes[2].set_title('Davies-Bouldin Score (min = meilleur)')
axes[2].set_xlabel('k')
axes[2].legend()

plt.tight_layout()
plt.savefig('../reports/figures/ex2_kmeans_selection.png', dpi=150, bbox_inches='tight')
plt.show()

BEST_K = list(k_range)[np.argmax(silhouette_scores)]
print(f"k optimal (Silhouette) : {BEST_K}")

In [ ]:
# K-Means final
kmeans = KMeans(n_clusters=BEST_K, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans.fit_predict(X_pca_full)

# Clustering hiérarchique
agglo = AgglomerativeClustering(n_clusters=BEST_K, linkage='ward')
agglo_labels = agglo.fit_predict(X_pca_full)

# GMM
gmm = GaussianMixture(n_components=BEST_K, random_state=RANDOM_STATE, covariance_type='full')
gmm_labels = gmm.fit_predict(X_pca_full)

# DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=10)
dbscan_labels = dbscan.fit_predict(X_pca_full)
n_dbscan_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
print(f"DBSCAN : {n_dbscan_clusters} clusters, {(dbscan_labels == -1).sum()} outliers")

In [ ]:
# Comparaison visuelle des algorithmes
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

algo_results = [
    (kmeans_labels, 'K-Means', axes[0, 0]),
    (agglo_labels, 'Agglomerative', axes[0, 1]),
    (gmm_labels, 'GMM', axes[1, 0]),
    (dbscan_labels, 'DBSCAN', axes[1, 1]),
]

for labels, title, ax in algo_results:
    scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=labels,
                         cmap='tab10', alpha=0.6, s=15)
    ax.set_title(title)
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
    plt.colorbar(scatter, ax=ax)

plt.suptitle('Comparaison des algorithmes de clustering', y=1.01, fontsize=14)
plt.tight_layout()
plt.savefig('../reports/figures/ex2_clustering_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Dendrogramme (clustering hiérarchique)
sample_idx = np.random.choice(len(X_pca_full), 300, replace=False)
Z = linkage(X_pca_full[sample_idx], method='ward')

plt.figure(figsize=(16, 6))
dendrogram(Z, truncate_mode='lastp', p=30, leaf_rotation=45, leaf_font_size=10)
plt.title('Dendrogramme (échantillon 300 clients)')
plt.xlabel('Clients')
plt.ylabel('Distance Ward')
plt.tight_layout()
plt.savefig('../reports/figures/ex2_dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Évaluation & comparaison

In [ ]:
# Tableau de comparaison
eval_results = []
valid_algos = [
    ('K-Means', kmeans_labels),
    ('Agglomerative', agglo_labels),
    ('GMM', gmm_labels),
]
if n_dbscan_clusters > 1:
    mask = dbscan_labels != -1
    valid_algos.append(('DBSCAN', dbscan_labels))

for name, labels in valid_algos:
    try:
        sil = silhouette_score(X_pca_full, labels)
        db = davies_bouldin_score(X_pca_full, labels)
        n_clust = len(set(labels)) - (1 if -1 in labels else 0)
        eval_results.append({'Algorithme': name, 'N clusters': n_clust,
                              'Silhouette ↑': round(sil, 4), 'Davies-Bouldin ↓': round(db, 4)})
    except Exception as e:
        print(f"{name} : {e}")

eval_df = pd.DataFrame(eval_results).set_index('Algorithme')
print(eval_df.to_string())

## 6. Interprétation métier des profils

In [ ]:
# Ajouter les labels K-Means au dataframe original
df_result = df_clean.copy()
df_result['Cluster'] = kmeans_labels

# Profil moyen par cluster
profile_cols = ['Income', 'Age', 'TotalSpend', 'TotalPurchases',
                'MntWines', 'MntMeatProducts', 'NumWebPurchases',
                'NumStorePurchases', 'Recency', 'Children']
profile_cols = [c for c in profile_cols if c in df_result.columns]

cluster_profiles = df_result.groupby('Cluster')[profile_cols].mean().round(1)
print("=== Profils moyens par cluster ===")
print(cluster_profiles.T.to_string())

In [ ]:
# Heatmap des profils normalisés
from sklearn.preprocessing import MinMaxScaler

profile_norm = pd.DataFrame(
    MinMaxScaler().fit_transform(cluster_profiles),
    index=cluster_profiles.index,
    columns=cluster_profiles.columns
)

plt.figure(figsize=(14, 6))
sns.heatmap(profile_norm.T, annot=cluster_profiles.T.round(0).astype(int),
            fmt='d', cmap='YlOrRd', linewidths=0.5, cbar_kws={'label': 'Valeur normalisée'})
plt.title('Profils clients par cluster (normalisés)', fontsize=14)
plt.xlabel('Cluster')
plt.tight_layout()
plt.savefig('../reports/figures/ex2_cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Nommer les clusters (à adapter selon les résultats)
# Exemple de nommage basé sur les profils observés :
cluster_names = {
    0: 'Clients Premium',       # Hauts revenus, fortes dépenses
    1: 'Clients Digitaux',      # Achats web élevés, jeunes
    2: 'Promo-sensibles',       # Nombreux achats promotionnels
    3: 'Clients Dormants',      # Faibles dépenses, peu actifs
}
# Adapter cluster_names selon BEST_K et les profils réels
if BEST_K <= len(cluster_names):
    df_result['Profil'] = df_result['Cluster'].map(
        {k: v for k, v in cluster_names.items() if k < BEST_K}
    )

print("Taille des clusters :")
print(df_result['Cluster'].value_counts().sort_index())

In [ ]:
# Radar chart des profils
radar_cols = ['Income', 'TotalSpend', 'NumWebPurchases', 'NumStorePurchases',
              'NumDealsPurchases', 'Recency']
radar_cols = [c for c in radar_cols if c in cluster_profiles.columns]

from matplotlib.patches import FancyArrowPatch
import matplotlib.patches as mpatches

N = len(radar_cols)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, axes = plt.subplots(1, min(BEST_K, 4), figsize=(16, 5), subplot_kw=dict(polar=True))
if BEST_K == 1:
    axes = [axes]

colors = plt.cm.tab10.colors
norm_profiles = MinMaxScaler().fit_transform(cluster_profiles[radar_cols])

for i, (ax, row) in enumerate(zip(axes[:BEST_K], norm_profiles)):
    values = row.tolist() + row[:1].tolist()
    ax.plot(angles, values, color=colors[i], linewidth=2)
    ax.fill(angles, values, color=colors[i], alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(radar_cols, size=8)
    ax.set_title(cluster_names.get(i, f'Cluster {i}'), size=10, pad=15)
    ax.set_ylim(0, 1)

plt.suptitle('Radar — Profils clients', y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig('../reports/figures/ex2_radar_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Recommandations business

In [ ]:
# Taux de réponse aux campagnes par cluster
cmp_cols = [c for c in df_result.columns if c.startswith('AcceptedCmp') or c == 'Response']

if cmp_cols:
    df_result['CampaignResponse'] = df_result[cmp_cols].sum(axis=1)
    campaign_by_cluster = df_result.groupby('Cluster')['CampaignResponse'].mean()

    fig, ax = plt.subplots(figsize=(10, 5))
    campaign_by_cluster.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
    ax.set_title('Réponse moyenne aux campagnes marketing par cluster')
    ax.set_xlabel('Cluster')
    ax.set_ylabel('Taux de réponse moyen')
    ax.tick_params(axis='x', rotation=0)
    plt.tight_layout()
    plt.savefig('../reports/figures/ex2_campaign_response.png', dpi=150, bbox_inches='tight')
    plt.show()

## Synthèse Exercice 2 & Recommandations

### Profils identifiés *(à compléter selon les résultats)*

| Cluster | Profil | Caractéristiques | Actions recommandées |
|---------|--------|-----------------|---------------------|
| 0 | **Clients Premium** | Hauts revenus, fortes dépenses vins/viande | Programme fidélité exclusif, offres premium |
| 1 | **Clients Digitaux** | Acheteurs web, visites fréquentes | Campagnes email, app mobile, livraison rapide |
| 2 | **Promo-sensibles** | Achats via deals, budget moyen | Promotions ciblées, bons de réduction |
| 3 | **Clients Dormants** | Faibles dépenses, longue récence | Campagne réactivation, offres découverte |

### Recommandations stratégiques
- **Fidélisation** : concentrer les efforts sur les Clients Premium (valeur à vie élevée)
- **Réactivation** : campagne personnalisée pour les Clients Dormants avec offre d'entrée
- **Conversion** : accompagner les Promo-sensibles vers des achats sans promotion
- **Digital** : développer l'expérience mobile/web pour les Clients Digitaux

### Métriques de suivi
- Recalculer les clusters tous les 6 mois (dérive comportementale)
- Suivre le Silhouette Score dans le temps
- Mesurer le taux de réponse par profil pour chaque campagne